<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_01_contingency_set.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 · Notebook 01 — The Contingency Set

**Paired with L12.2 · Prediction of Power Grid Stability**

Notebook 00 computed one critical clearing time and timed it. This notebook
computes one thousand and eighty of them, which is what a supervised model
needs and what makes the case for not doing it again.

Three things happen here.

1. **The dataset gets built.** 180 plausible dispatches crossed with the 6
   contingencies. This is the slow cell — about five and a half minutes — and
   it caches, so you pay once.
2. **The class balance gets looked at, before any model is trained.** A
   contingency set that is 95 % secure trains a predictor that says "secure"
   and scores 95 %. Knowing the balance first is what stops you being pleased
   by that.
3. **You label one case by hand**, so that "ground truth" is something you have
   produced rather than something that arrived in an array.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
import os, time
os.makedirs(pb.RESULTS, exist_ok=True)

cases = pb.contingencies()
print(f"{pb.N_CONTINGENCY} contingencies, "
      f"{len(pb.islanding_outages())} branch dropped as islanding")
print(f"protection time : {pb.PROTECTION_TIME*1e3:.0f} ms "
      f"({pb.PROTECTION_TIME*50:.0f} cycles at 50 Hz)")
print(f"bisection ceiling: {pb.CCT_MAX*1e3:.0f} ms")

---

## 1 · The operating points

A dataset of contingencies at one dispatch would teach a model nothing: the
answer would be a lookup table with six entries. What makes screening a
*prediction* problem is that the operating point moves, and moves the answer
with it.

`operating_points` draws three independent knobs — system load, the split
between local generation and import, and the German HVDC link — plus a small
independent jitter on each load bus so the three load centres do not move in
lockstep. The ranges and the reasoning are in its docstring; read it before you
quote any of the numbers below.

In [ ]:
ops = pb.operating_points(12, seed=1)

print("shape:", ops.shape, "  (n, [P;Q], bus)")
print()
print(f"  {'':4s}{'load total':>12s}{'Gen east':>11s}{'HVDC':>9s}"
      f"{'slack import':>15s}")
for i, o in enumerate(ops):
    V, th, ok, _ = pb.solve_power_flow(pb.build_ybus(), o[0], o[1])
    Pc, _ = pb.pq_from_state(V, th, pb.build_ybus())
    print(f"  {i:<4d}{-o[0][[2,3,4]].sum():>12.3f}{o[0][1]:>11.3f}"
          f"{o[0][5]:>9.3f}{Pc[0]:>15.3f}")

print()
print(f"  base case for comparison: load {0.58+1.05+0.46:.3f},"
      f" Gen east 1.150, HVDC 0.360")

**What you should see.** Twelve dispatches with total load between roughly 1.7
and 3.5 p.u. against a base of 2.09, Gen east between about 0.8 and 2.3 p.u.,
and a slack import that swings both ways — negative entries are Zealand
exporting to Sweden, which happens on a windy night.

That range is wider than a single day's excursion, deliberately. A screening
surrogate that has only ever seen the middle of the operating range is exactly
the surrogate that fails on the evening it is needed.

---

## 2 · Build the labelled set

Every operating point is crossed with every contingency: `180 × 6 = 1080`
labels, each one a power flow, three Kron reductions, an equilibrium solve and
about eleven RK4 integrations of the swing equations.

**This cell takes five to six minutes the first time.** It then caches to
`Ex12.2_outputs/dataset_n180_s12.npz` and returns instantly on every rerun. If
you are short of time, `n_ops=60` runs in under two minutes and everything
downstream still works — say so in your report if you do.

In [ ]:
t0 = time.time()
data = pb.build_dataset(n_ops=180, seed=12)
print(f"\n  build_dataset returned in {time.time()-t0:.1f} s"
      f"  (instant if it came from the cache)")

X       = data["X"]          # (n, 6 buses, 6 features)
A       = data["A"]          # (n, 6, 6) post-fault adjacency
onehot  = data["onehot"]     # (n, 6) which branch is out
cct     = data["cct"]        # (n,) seconds
op_id   = data["op_id"]
cont_id = data["cont_id"]

check_shape("node features", X, (1080, 6, 6))
check_shape("adjacency", A, (1080, 6, 6))
check_shape("one-hot", onehot, (1080, 6))
check_shape("labels", cct, (1080,))
print()
print("feature channels:", pb.FEATURE_NAMES)

**What you should see.** Four PASSes, and — the first time — a progress trace
ending in something close to

```
  1080 labels in 335.1 s (310 ms each)
  CCT  min 0.056   median 0.227   max 0.500  s
  insecure at 140 ms: 175 of 1080 (16.2 %)
  censored at CCT_MAX = 0.50 s: 186 (17.2 %)
```

Your seconds will differ with your machine; the labels will not, because the
seed is fixed and every step of the pipeline is deterministic.

**310 ms per label.** That is the number the rest of this exercise set is
arguing with.

---

## 3 · Class balance, before anything is trained

Look at this now, not after your first model scores 88 %.

In [ ]:
insecure = cct <= pb.PROTECTION_TIME

print(f"  cases            : {len(cct)}")
print(f"  insecure         : {insecure.sum():4d}   ({100*insecure.mean():.1f} %)")
print(f"  secure           : {(~insecure).sum():4d}   ({100*(~insecure).mean():.1f} %)")
print(f"  censored at {pb.CCT_MAX*1e3:.0f} ms: {(cct >= pb.CCT_MAX).sum():4d}"
      f"   ({100*(cct >= pb.CCT_MAX).mean():.1f} %)")
print()
print(f"  CCT  mean {cct.mean()*1e3:6.1f} ms   std {cct.std()*1e3:6.1f} ms")
print(f"       min  {cct.min()*1e3:6.1f} ms   median {np.median(cct)*1e3:6.1f} ms"
      f"   max {cct.max()*1e3:6.1f} ms")
print()
print("  ACCURACY OF A MODEL THAT ALWAYS SAYS 'SECURE':"
      f"  {100*(~insecure).mean():.1f} %")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
pb.use_course_style()
ax[0].hist(cct*1e3, bins=40, color=pb.CYAN)
ax[0].axvline(pb.PROTECTION_TIME*1e3, color=pb.ORANGE, lw=1.6, ls="--",
              label=f"protection {pb.PROTECTION_TIME*1e3:.0f} ms")
ax[0].axvline(pb.CCT_MAX*1e3, color=pb.MUTED, lw=1.2, ls=":", label="censoring ceiling")
ax[0].set_xlabel("critical clearing time  (ms)"); ax[0].set_ylabel("cases")
ax[0].legend(fontsize=8); ax[0].set_title("all 1080 labels", fontsize=10)

ax[1].hist(cct[cct < pb.CCT_MAX]*1e3, bins=40, color=pb.GREEN)
ax[1].axvline(pb.PROTECTION_TIME*1e3, color=pb.ORANGE, lw=1.6, ls="--")
ax[1].set_xlabel("critical clearing time  (ms)")
ax[1].set_title("uncensored cases only", fontsize=10)
fig.tight_layout(); plt.show()

**What you should see.** **175 insecure of 1080 — 16.2 %.** So a model that
prints "secure" for everything and is never wrong about anything else scores
**83.8 %**, and 83.8 % is therefore not a result. Any accuracy you report from
here on has to be read against that number, and notebook 04 stops reporting
accuracy at all for exactly this reason.

The histogram has a spike at 500 ms — **186 cases, 17.2 %** — where the
bisection stopped. Those are not measurements of anything; they are the
statement "longer than half a second, and we stopped caring". A regression
trained on them as if they were measurements will spend capacity fitting a
number that does not exist. Three defensible things to do about it, in
increasing order of effort:

* leave them in and **report regression error separately for the uncensored
  subset**, which is what notebooks 02 and 03 do;
* clip the target and admit the model is now predicting `min(CCT, 500 ms)`;
* drop them, and accept that you have removed the easy cases and made every
  score look worse.

None of those is wrong. Quietly doing the first and reporting a single MAE
over everything is.

---

## 4 · Which contingencies are dangerous, and why

This is the table a planner would actually read.

In [ ]:
rows = []
for c in cases:
    m = cont_id == c["index"]
    rows.append([c["label"],
                 f"{np.median(cct[m])*1e3:.1f}",
                 f"{cct[m].min()*1e3:.1f}",
                 f"{cct[m].max()*1e3:.1f}",
                 f"{100*(cct[m] <= pb.PROTECTION_TIME).mean():.1f} %",
                 f"{100*(cct[m] >= pb.CCT_MAX).mean():.1f} %"])
print(error_table(rows, ["contingency", "median CCT [ms]", "min", "max",
                         "insecure", "censored"]))

worst = np.array([cont_id[op_id == i][np.argmin(cct[op_id == i])]
                  for i in range(data["ops"].shape[0])])
print()
print("  for each dispatch, which contingency is the WORST:")
for c in cases:
    print(f"    {c['label']:<46s} {int((worst == c['index']).sum()):4d} of "
          f"{data['ops'].shape[0]}")

per_op_min = np.array([cct[op_id == i].min() for i in range(data["ops"].shape[0])])
print()
print(f"  dispatches with at least one insecure contingency: "
      f"{int((per_op_min <= pb.PROTECTION_TIME).sum())} of "
      f"{data['ops'].shape[0]}"
      f"  ({100*(per_op_min <= pb.PROTECTION_TIME).mean():.0f} %)")

**What you should see.** A table with a very clear shape:

| contingency | median CCT | insecure |
|---|---|---|
| base — fault at Gen east, no line lost | 182.6 ms | 19.4 % |
| trip line 0 (Slack – Gen east) | **154.3 ms** | **40.0 %** |
| trip line 1 (Gen east – Load north) | 182.6 ms | 19.4 % |
| trip line 2 (Load north – Load city) | 386.7 ms | 0.0 % |
| trip line 3 (Slack – Load south) | 187.5 ms | 18.3 % |
| trip line 4 (Load south – Load city) | 500.0 ms | 0.0 % |

and, underneath it, **line 0 is the worst contingency for all 180 dispatches**,
while **40 % of dispatches have at least one insecure contingency**.

The physics behind that table is worth stating, because a model that gets it
right for the wrong reason is not a model you can deploy.

**Where the fault is decides the severity.** Contingencies 0, 1 and 2 put the
fault at or next to a machine bus — Gen east's terminals, or the slack. During
the fault the transfer admittance collapses to a hundredth of its value, the
machine sees almost no electrical load, and it accelerates on its full
mechanical power. Contingencies 3 and 5 put the fault two hops away at a load
bus, where the direct Slack–Gen east tie carries on regardless, and the machine
barely notices. Those two never go insecure at any dispatch in this set.

**What the post-fault network decides is whether the energy can be given back.**
This is why line 0 is the worst of them. It is the only branch whose loss
lengthens the electrical distance between the two machines: with it gone, the
path from Gen east to the Swedish system runs the long way round through Load
north and Load city. The machine picked up the same kinetic energy during the
fault and now has a weaker network to shed it into.

Notice what that argument used. It used **structure**: which bus, how many
hops, which path survives. It did not use the branch's index. A model handed
the topology can in principle reason the same way. A model handed a one-hot flag
cannot; it can only memorise that bit number 0 means trouble.

In [ ]:
sel = op_id == 0
pb.plot_screening(cct[sel], labels=[cases[k]["label"] for k in cont_id[sel]],
                  title="One dispatch, all six contingencies, ranked")
plt.tight_layout(); plt.show()

---

## 5 · What moves the clearing time

Two variables, both of which an operator controls.

In [ ]:
total_load = -data["ops"][:, 0, [2, 3, 4]].sum(axis=1)[op_id]
p_gen = data["ops"][:, 0, 1][op_id]

print(f"  corr(CCT, total load)   {np.corrcoef(cct, total_load)[0,1]:+.3f}")
print(f"  corr(CCT, Gen east P)   {np.corrcoef(cct, p_gen)[0,1]:+.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
pb.use_course_style()
for k, c in enumerate(cases):
    m = cont_id == c["index"]
    ax[0].plot(p_gen[m], cct[m]*1e3, "o", ms=3, alpha=0.55,
               color=CYCLE[k % len(CYCLE)], label=f"cont {c['index']}")
ax[0].axhline(pb.PROTECTION_TIME*1e3, color=pb.ORANGE, lw=1.4, ls="--")
ax[0].set_xlabel("Gen east dispatch  (p.u.)"); ax[0].set_ylabel("CCT  (ms)")
ax[0].legend(fontsize=7, ncol=2); ax[0].set_title("loading against margin", fontsize=10)

m0 = cont_id == 1
ax[1].plot(cct[cont_id == 0]*1e3, cct[m0]*1e3, "o", ms=4, alpha=0.6, color=pb.GREEN)
lim = [0, 1.05*max(cct[cont_id == 0].max(), cct[m0].max())*1e3]
ax[1].plot(lim, lim, "-", color=pb.MUTED, lw=1.0)
ax[1].axhline(pb.PROTECTION_TIME*1e3, color=pb.ORANGE, lw=1.2, ls="--")
ax[1].axvline(pb.PROTECTION_TIME*1e3, color=pb.ORANGE, lw=1.2, ls="--")
ax[1].set_xlim(lim); ax[1].set_ylim(lim)
ax[1].set_xlabel("base case CCT  (ms)"); ax[1].set_ylabel("line 0 out, CCT  (ms)")
ax[1].set_title("what losing the tie costs", fontsize=10)
fig.tight_layout(); plt.show()

**What you should see.** Correlations of about **−0.49** with total load and
**−0.50** with the Gen east dispatch: load the machine and its margin falls.
That is the expected sign and the expected strength — it is a strong effect but
it explains only a quarter of the variance, because *which* contingency you are
asking about matters at least as much.

The right-hand panel is the one to keep. Every point is one dispatch, plotted as
(base-case CCT, line-0-out CCT). Every point sits **below** the diagonal: losing
the tie always costs margin. And a band of points sits below the horizontal
threshold while remaining to the right of the vertical one — dispatches that are
perfectly safe as they stand and insecure the moment one line is lost. **That
band is what N-1 screening exists to find.**

---

## 6 · Your turn: label one case by hand

`build_dataset` handed you an array. Before you train anything on it, produce
one of its entries yourself, from the pieces, and check that it agrees.

### Your turn

In [ ]:
# TODO: reproduce one label of the dataset from first principles.
#
#   Pick a row of the dataset -- say the most stressed case there is:
#
#       row  = int(np.argmin(cct))
#       i_op = op_id[row]
#       cse  = cases[cont_id[row]]
#       op   = data["ops"][i_op]
#
#   Now build it yourself with pb.case_setup, and read off the pieces:
#
#       detail = pb.case_setup(op, cse["outage"], cse["fault_bus"])
#       delta0 = detail["delta0"]
#       y_pre, y_flt, y_pst = (abs(detail["Yred_pre"][0, 1]),
#                              abs(detail["Yred_fault"][0, 1]),
#                              abs(detail["Yred_post"][0, 1]))
#
#   Then find the clearing time by hand rather than by bisection: simulate at
#   a few clearing times and see where the machine stops coming back.
#
#       trials = np.arange(0.02, 0.20, 0.01)
#       for tc in trials:
#           T, D, W = pb.simulate_swing(detail["machines"], detail["Yred_pre"],
#                                       detail["Yred_post"], t_end=pb.T_END,
#                                       t_fault=pb.T_FAULT, t_clear=pb.T_FAULT+tc,
#                                       Yred_fault=detail["Yred_fault"],
#                                       delta0=detail["delta0"])
#           sep = np.abs(D[:, 1] - D[:, 0]).max()          # radians
#           ... record (tc, np.degrees(sep), sep < np.pi)
#
#   Record, by these names, because the next cell uses them:
#       row                    -- the dataset row you chose
#       detail                 -- the dict from pb.case_setup
#       y_pre, y_flt, y_pst    -- the three transfer admittance magnitudes
#       my_cct                 -- the largest tc in your sweep that stayed stable
#       ref_cct                -- cct[row], the dataset's answer
#
# Your sweep steps in 10 ms and the dataset bisected to 2 ms, so the two will
# not be equal. They must be within one step of each other, and my_cct must be
# the SMALLER: a coarse sweep can only underestimate the boundary.

raise NotImplementedError("Label one case by hand and compare with the dataset")

In [ ]:
print(f"  contingency        : {cases[cont_id[row]]['label']}")
print(f"  your sweep         : {my_cct*1e3:6.1f} ms")
print(f"  dataset (bisected) : {ref_cct*1e3:6.1f} ms")
print(f"  difference         : {(ref_cct-my_cct)*1e3:6.1f} ms"
      f"   (one sweep step is 10.0 ms)")
print(f"  agree within a step: {0 <= (ref_cct - my_cct) < 0.010 + 1e-9}")
print()
print(f"  |Y_transfer|  pre {y_pre:.4f}   fault {y_flt:.6f}   post {y_pst:.4f}")

T, D, W = pb.simulate_swing(detail["machines"], detail["Yred_pre"],
                            detail["Yred_post"], t_end=pb.T_END,
                            t_fault=pb.T_FAULT, t_clear=pb.T_FAULT + my_cct,
                            Yred_fault=detail["Yred_fault"],
                            delta0=detail["delta0"])
pb.plot_swing(T, D, W, t_fault=pb.T_FAULT, t_clear=pb.T_FAULT + my_cct,
              title=f"the hardest case in the set, cleared at {my_cct*1e3:.0f} ms")
plt.show()

**What you should see.** The hardest case in the dataset has a true CCT of
**55.7 ms — under three cycles**, and it is contingency 1, the tie outage, at
one of the heaviest dispatches. Your hand sweep in 10 ms steps should land at
50 ms: below the true value, within one step, never above.

If your number came out *above* the dataset's, you have a bug, and it is almost
certainly that you recorded the first unstable `tc` rather than the last stable
one.

---

## 7 · The split, and why it is not random

The 1080 rows are not 1080 independent cases. Six of them share an operating
point, and therefore share a power flow, a set of voltages and five of the six
feature channels. Split those six across training and test and the model can
score well by recognising the dispatch rather than by understanding the
contingency.

So the split is on `op_id`, always.

In [ ]:
n_train_ops = 144                      # 80 % of 180
train = op_id < n_train_ops
test = ~train

print(f"  train : {train.sum():4d} cases from {n_train_ops} dispatches")
print(f"  test  : {test.sum():4d} cases from "
      f"{data['ops'].shape[0]-n_train_ops} dispatches")
print(f"  no dispatch appears in both: "
      f"{len(set(op_id[train]) & set(op_id[test])) == 0}")
print()
print(f"  insecure fraction, train {100*(cct[train] <= pb.PROTECTION_TIME).mean():.1f} %")
print(f"  insecure fraction, test  {100*(cct[test] <= pb.PROTECTION_TIME).mean():.1f} %")

pb.save("nb01_split", n_train_ops=np.array(n_train_ops),
        train=train, test=test, cct=cct, cont_id=cont_id, op_id=op_id)

**What you should see.** 864 training cases and 216 test cases, no dispatch in
both, and insecure fractions of **17.2 %** and **12.0 %**.

Those two are not equal, and they should not be expected to be: 36 dispatches
is a small sample and the insecure cases cluster in the heavily loaded ones. It
is worth noticing before notebook 02 reports a test accuracy, because a
five-point difference in base rate moves an accuracy figure by about that much
on its own.

---

## 8 · Before you move on

1. A model that always predicts "secure" scores 83.8 % on this set. Name the
   metric you would report instead, and say what it is worth when the insecure
   class is 16 % of the data.
2. 17.2 % of the labels are censored at the bisection ceiling. State what you
   will do about them, and what that choice makes your regression error a
   measurement *of*.
3. Contingency 5 — trip the line from Load south to Load city — never went
   insecure at any of the 180 dispatches. Give one change to the operating
   range that would make it go insecure, and say whether that change is
   realistic for eastern Denmark.
4. The split is on `op_id`. Describe, in one sentence, the number you would have
   reported if you had split at random instead, and why it would have been
   larger.

Next: **notebook 02**, the dense baseline — flatten everything, name the outage
with a flag, and see how far that gets.